# Forecast-period surface energy budget

Member-level daily surface-energy-budget terms for the hindcasts, cached per member, and regional ensemble time series of each term. Run once with `max_members = 1` as a smoke test before processing every member.

Cases: **S0** hindcasts start from the Dec 2011 analyses and cover Jan-Feb 2012; **S1** hindcasts start from the May 2012 analyses and cover Jun-Jul 2012. Each case compares hindcasts initialized from CTRL, CAPT and EAM-DART states.

## 1. Settings

In [ ]:
from pathlib import Path

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
LAND_SUPPORT_DIR = Path("/compyfs/zhan391/acme_init/lnd_sea_mask")  # land mask
workflow_name = "fcst_atm_model_evaluation"

run_segment = "fc"
# Forecast cases: season -> experiments compared together (same period).
cases = {
    "S0": ("CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0"),  # 201201-201202
    "S1": ("CTRL10-S1", "CAPT10-S1", "DART40-S1"),  # 201206-201207
}
plot_variables = ("Rn", "LE", "SH", "Residual", "BowenRatio")
region = "land"
max_members = None  # e.g. 1 for a smoke test
force_compute = False
show = True
FIGURE_DPI = 200

## 2. Project setup

In [ ]:
import sys
import matplotlib.pyplot as plt

repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.bias_metrics import experiment_registry
from util.atm_surface_energy_budget import OUTPUT_VARIABLES, REGIONS, cache_surface_energy_budget, energy_budget_cache_path, plot_energy_budget_timeseries

data_dir, figure_dir = workflow_output_dirs(
    workflow_name, "surface_energy_budget", output_root=DIAGNOSTIC_OUTPUT_ROOT
)
if region not in REGIONS:
    raise ValueError(f"Unknown region {region!r}; choose from {list(REGIONS)}")
unknown = set(plot_variables) - set(OUTPUT_VARIABLES)
if unknown:
    raise ValueError(f"Unknown variables {sorted(unknown)}; choose from {list(OUTPUT_VARIABLES)}")
landmask = LAND_SUPPORT_DIR / "landmask_1x1.nc"

## 3. Compute or reuse member budgets

In [ ]:
registries, processed = {}, {}
for season, experiments in cases.items():
    registry = registries[season] = experiment_registry(INPUT_DATA_ROOT, season, run_segment)
    unknown = set(experiments) - set(registry)
    if unknown:
        raise ValueError(f"{season}: {sorted(unknown)} unavailable; choose from {list(registry)}")
    for experiment in experiments:
        meta = registry[experiment]
        count = meta["nens"] if max_members is None else min(max_members, meta["nens"])
        members = {}
        for number in range(1, count + 1):
            member = f"EN{number:02d}"
            target = energy_budget_cache_path(data_dir, experiment, member, meta["period"])
            existed = target.is_file()
            members[member] = cache_surface_energy_budget(
                target, data_root=INPUT_DATA_ROOT, experiment=experiment,
                season=season, run_segment=run_segment, member=member,
                period=meta["period"], force_compute=force_compute,
            )
            print("[REUSE]" if existed and not force_compute else "[WRITE]", target)
        processed.setdefault(season, {})[experiment] = members

## 4. Plot, save, and show

In [ ]:
for season, by_experiment in processed.items():
    registry = registries[season]
    for variable in plot_variables:
        fig = plot_energy_budget_timeseries(
            by_experiment, variable, region, landmask,
            label_map={name: registry[name]["label"] for name in by_experiment},
        )
        output = figure_dir / f"{variable}_{region}_{season}_{run_segment}.png"
        fig.savefig(output, dpi=FIGURE_DPI, bbox_inches="tight")
        print(f"[FIGURE] {output}")
        if not show:
            plt.close(fig)

## Notes: variables and regions

Required model inputs: `LHFLX` (or `QFLX`, converted using the latent heat of vaporization), `SHFLX`, `FSNS`, and `FLNS`. Cached derived variables: `Rn = FSNS - FLNS`, `LE`, `SH`, `Residual = Rn - (LE + SH)`, and `BowenRatio = SH / LE`.

Plot regions: `global`, `land`, and `ocean`. Forecast runs are continuous, so the budget reads daily files and falls back to 6-hourly data.

Caches: one file per experiment/member/period under `surface_energy_budget/`.